# Store Demand Quality Lab

A standalone, reproducible retail demand notebook for a small store-opening panel. It emphasizes data quality, cold-start history, time-aware validation, WMAPE, and bias.

## 1. Setup and portable data loading

Place a compatible CSV at `data/demand_daily.csv` to use real data. Without one, the notebook generates the same 10-store opening pattern described in the interview brief.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RANDOM_SEED = 42
EXPECTED_COLUMNS = ["date", "store", "sales", "deliveries", "spoilage", "promotion_type"]
DATA_PATH = Path("data/demand_daily.csv")


def make_demo_data(seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    opening_dates = {
        "store_01": "2024-01-01", "store_02": "2024-01-01",
        "store_03": "2024-01-01", "store_04": "2024-01-01",
        "store_05": "2024-04-01", "store_06": "2024-04-01",
        "store_07": "2024-09-01", "store_08": "2024-09-01",
        "store_09": "2025-02-01", "store_10": "2025-02-01",
    }
    rows = []
    for store, opening_date in opening_dates.items():
        dates = pd.date_range(opening_date, "2025-12-31", freq="D")
        store_index = int(store[-2:])
        seasonal = 8 * np.sin(2 * np.pi * dates.dayofyear.to_numpy() / 365.25)
        weekly = np.where(dates.dayofweek.to_numpy() >= 4, 7, -2)
        promotion = np.where((dates.dayofweek.to_numpy() == 5) & (dates.day.to_numpy() % 3 == 0), "discount", "none")
        expected_sales = 78 + store_index * 2 + seasonal + weekly + np.where(promotion == "discount", 18, 0)
        sales = np.maximum(0, rng.poisson(np.maximum(expected_sales, 1)))
        deliveries = sales + rng.integers(12, 35, len(dates))
        spoilage = np.maximum(0, deliveries - sales - rng.integers(0, 12, len(dates)))
        rows.extend(zip(dates, [store] * len(dates), sales, deliveries, spoilage, promotion))
    return pd.DataFrame(rows, columns=EXPECTED_COLUMNS)


def load_and_clean(path=DATA_PATH):
    source = "demo generator"
    raw = make_demo_data()
    if path.exists():
        raw = pd.read_csv(path)
        source = str(path)
    missing = sorted(set(EXPECTED_COLUMNS) - set(raw.columns))
    if missing:
        raise ValueError(f"Missing required columns: {missing}")
    frame = raw[EXPECTED_COLUMNS].copy()
    frame["date"] = pd.to_datetime(frame["date"], errors="coerce")
    numeric = ["sales", "deliveries", "spoilage"]
    frame[numeric] = frame[numeric].apply(pd.to_numeric, errors="coerce")
    frame = frame.dropna(subset=["date", "store", *numeric]).copy()
    invalid = ((frame["sales"] < 0) | (frame["sales"] > 500) | (frame["deliveries"] < 0) | (frame["spoilage"] < 0))
    removed_rows = int(invalid.sum())
    frame = frame.loc[~invalid].sort_values(["store", "date"]).reset_index(drop=True)
    frame["promotion_type"] = frame["promotion_type"].fillna("unknown").astype(str)
    return frame, source, removed_rows


df, data_source, removed_rows = load_and_clean()
print(f"Source: {data_source}")
print(f"Rows: {len(df):,} | columns: {len(df.columns)} | removed invalid rows: {removed_rows}")
df.head()

## 2. Size, store openings, and continuity

The missing store-days are structural: stores open in waves, so a per-store model must not mistake a short history for missing observations.

In [ ]:
history = (
    df.groupby("store")
    .agg(first_date=("date", "min"), last_date=("date", "max"), rows=("date", "size"))
    .assign(calendar_days=lambda x: (x["last_date"] - x["first_date"]).dt.days + 1)
    .assign(completeness=lambda x: x["rows"] / x["calendar_days"])
    .sort_values("first_date")
)
history["days_available"] = history["calendar_days"]
print(f"Date range: {df.date.min().date()} to {df.date.max().date()}")
print(f"Stores: {df.store.nunique()} | rows: {len(df):,} | possible store-days over full range: {df.store.nunique() * ((df.date.max() - df.date.min()).days + 1):,}")
discontinuities = []
for store, group in df.groupby("store"):
    gaps = group["date"].sort_values().diff().dropna().dt.days
    discontinuities.append({"store": store, "max_gap_days": int(gaps.max()) if len(gaps) else 0, "duplicate_dates": int(group.date.duplicated().sum())})
continuity = pd.DataFrame(discontinuities).set_index("store")
print("Internal gaps:", int((continuity.max_gap_days > 1).sum()), "| duplicate store-dates:", int(continuity.duplicate_dates.sum()))
history

## 3. Data quality checks

Negative quantities are removed rather than imputed because they are impossible operational values. Extreme positive sales are flagged by the explicit threshold in the loader and should be reviewed against the source system.

In [ ]:
quality = pd.Series({
    "null_cells": int(df.isna().sum().sum()),
    "negative_sales": int((df.sales < 0).sum()),
    "negative_deliveries": int((df.deliveries < 0).sum()),
    "negative_spoilage": int((df.spoilage < 0).sum()),
    "duplicate_store_dates": int(df.duplicated(["store", "date"]).sum()),
})
print(quality.to_string())
assert quality["null_cells"] == 0
assert quality["negative_sales"] == 0
assert quality["negative_deliveries"] == 0
assert quality["negative_spoilage"] == 0
assert quality["duplicate_store_dates"] == 0

## 4. A leakage-safe baseline

Use a 28-day, time-ordered holdout. The baseline is the previous 7-day observation for each store. WMAPE measures aggregate operational error; bias shows systematic over- or under-forecasting.

In [ ]:
def wmape(actual, forecast):
    denominator = np.abs(actual).sum()
    return float(np.abs(actual - forecast).sum() / denominator) if denominator else np.nan


def bias(actual, forecast):
    return float((forecast - actual).mean())


def add_features(group):
    group = group.sort_values("date").copy()
    group["lag_7"] = group["sales"].shift(7)
    group["rolling_7"] = group["sales"].shift(1).rolling(7, min_periods=7).mean()
    group["day_of_week"] = group["date"].dt.dayofweek
    return group


features = df.groupby("store", group_keys=False).apply(add_features)
features = features.dropna(subset=["lag_7", "rolling_7"])
holdout_start = features.date.max() - pd.Timedelta(days=27)
train = features[features.date < holdout_start]
test = features[features.date >= holdout_start].copy()
test["forecast_lag_7"] = test["lag_7"]
test["forecast_roll_7"] = test["rolling_7"]
results = pd.DataFrame({
    "model": ["seasonal_naive_7", "rolling_mean_7"],
    "WMAPE": [wmape(test.sales.to_numpy(), test.forecast_lag_7.to_numpy()), wmape(test.sales.to_numpy(), test.forecast_roll_7.to_numpy())],
    "bias_units": [bias(test.sales.to_numpy(), test.forecast_lag_7.to_numpy()), bias(test.sales.to_numpy(), test.forecast_roll_7.to_numpy())],
})
print(f"Train through: {train.date.max().date()} | holdout: {test.date.min().date()} to {test.date.max().date()}")
results

## 5. Visual diagnostic and conclusions

The newest stores have less history, so they should borrow signal from the chain or use a pooled model. Do not fit yearly seasonality independently when a store has not observed a full year.

In [ ]:
store_to_plot = history.index[-1]
plot_data = test[test.store == store_to_plot].sort_values("date")
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(plot_data.date, plot_data.sales, label="actual", linewidth=1.8)
ax.plot(plot_data.date, plot_data.forecast_lag_7, label="7-day seasonal naive", linewidth=1.4)
ax.set_title(f"28-day holdout: {store_to_plot}")
ax.set_ylabel("sales")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

print("Decision summary:")
print("- Validate in time order; never let future sales enter rolling features.")
print("- Report WMAPE and bias, not RMSE alone.")
print("- Pool short-history stores with chain-level or hierarchical features.")
print("- Treat deliveries and spoilage as operational constraints, not just extra predictors.")